# 19. Prospects

Forecast, Phase 1, step 2d. The season simulator needs **prospects**: drafted skaters and goalies who aren't NHL regulars yet, so a team can call one up when a player gets hurt or a kid makes the team. Our profiles (`data/bio_raw`) only hold players who already reached the NHL. This notebook:

1. **Downloads** the 2019 to 2026 draft classes and the profiles of every pick we don't have yet.
2. **Fixes how pre-NHL scoring is summarized.** Notebook 18 originally used each player's *best* pre-NHL season with 20+ games, which missed Matthew Schaefer's injury-shortened draft year. Four rules are compared on notebook 16's points backtest.
3. **Picks the best prior for players with no NHL time,** tested on notebook 18's true rookies.
4. **Projects every prospect for 2026-27**, with a range.

The new profiles go to `data/prospect_raw/`, **not** `data/bio_raw/`: adding non-NHL players there would change notebook 16's chained factors and notebooks 16-18 on a rerun. Both new folders are gitignored, and so is the result (`data/prospects_2026-27.csv`). `projections_2026-27.csv` isn't touched. Nothing on the cards, the app or the README changes.

The downloads take about 10-15 minutes the first time (a short pause between requests, to be polite to the NHL's servers). They're resumable: rerun the cell and only missing files are fetched.

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table, backtest, group_average, GOALS_PER_WIN
from edge_data import download_drafts, build_draft_table, download_bios, build_bio_table
from nhle import load_profiles, career_totals, EXCLUDED
from development import (training_nhle, best_nhle, nhle_rules, league_average_nhle, skater_features, fit_prior,
                         apply_prior, model_table, eligible, candidates, fit_rows, first_seasons, league_group,
                         prospect_table, prospect_range, wmean, wsd, TARGETS, AGING, SK_WEIGHTS, SK_REG,
                         PRIOR_INPUTS)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
OUT = DATA                      # where prospects_2026-27.csv is saved
RAW_DRAFT = DATA / "draft_raw"
RAW_PROSPECT = DATA / "prospect_raw"
PROFILE_LIMIT = None            # None = download every missing profile
YEARS = range(2019, 2027)

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

history = history_table(pd.read_csv(DATA / "war_by_season_2008_2025.csv"), pd.read_csv(DATA / "goalie_war_2008_2025.csv"))
skaters = history[history["group"] != "G"]
bios = build_bio_table(DATA / "bio_raw", 2026)
profiles = load_profiles(DATA / "bio_raw")
careers = pd.read_csv(DATA / "career_totals.csv")
nhle_factors = pd.read_csv(DATA / "nhle_factors.csv")
nhle_mult = pd.read_csv(DATA / "nhle_age_multipliers.csv")
dev_ranges = pd.read_csv(DATA / "dev_ranges.csv")
luck = dev_ranges[dev_ranges["kind"] == "luck_fit"].set_index("group")[["a", "b"]]
rookie_range = dev_ranges[dev_ranges["kind"] == "rookie"]
projections = pd.read_csv(DATA / "projections_2026-27.csv")

TRAIN_END = 2017
TRAIN = [t for t in TARGETS if t <= TRAIN_END]
TEST = [t for t in TARGETS if t > TRAIN_END]

## 2. Downloads: draft classes and missing profiles

The draft records (records.nhl.com) list every pick with the player's NHL id. For each pick not already in `bio_raw`, the player's NHL profile is downloaded (the same landing page as `bio_raw`: birth date, draft details and every season in every league).

In [ ]:
download_drafts(YEARS, RAW_DRAFT)
drafts = build_draft_table(RAW_DRAFT)
have = set(bios["playerId"])
drafts["in_bio_raw"] = drafts["playerId"].isin(have)
by_year = drafts.groupby("draft_year").agg(picks=("pick", "size"), in_bio_raw=("in_bio_raw", "sum"))
by_year["to download"] = by_year["picks"] - by_year["in_bio_raw"]
print(f"{len(drafts):,} picks, 2019 to 2026")
by_year

Draft years saved: 8
3 pick(s) without a playerId dropped
1,777 picks, 2019 to 2026


,picks,in_bio_raw,to download
draft_year,,,
2019,217,102,115
2020,216,95,121
2021,223,71,152
2022,225,76,149
2023,224,42,182
2024,225,20,205
2025,224,9,215
2026,223,0,223


In [ ]:
# Newest classes first, so a limited (test) run still covers the latest picks
new = drafts[~drafts["in_bio_raw"]].sort_values(["draft_year", "pick"], ascending=[False, True])["playerId"].drop_duplicates()
new_ids = list(new if PROFILE_LIMIT is None else new.head(PROFILE_LIMIT))
failed = download_bios(new_ids, RAW_PROSPECT)
print(f"Profiles in prospect_raw: {len(list(RAW_PROSPECT.glob('*.json'))):,}")

1361 already saved, 0 to download (about 0 minutes)
Done, no failures.
Profiles in prospect_raw: 1,361


## 3. Who is a prospect for 2026-27?

A prospect is:
- **drafted** 2019 to 2026,
- **not** in `projections_2026-27.csv` (so no NHL games from 2023-24 to 2025-26),
- **played** a regular-season game somewhere in 2025-26 (tournaments like the World Juniors don't count),
- **24 or under** on February 1, 2027.

Two tournament names show up only in prospect profiles (the NHL rookie "Prospects Challenge" and "International-Jr"), so they're excluded here on top of notebook 16's list. Notebook 16's list itself isn't changed (that would change notebooks 16-18).

**His team is the team that drafted him.** The data doesn't show rights trades, or rights that expired because a player was never signed, so a few prospects will be listed with the wrong team.

In [ ]:
prospect_profiles = load_profiles(RAW_PROSPECT) if any(RAW_PROSPECT.glob("*.json")) else profiles.iloc[:0]
all_profiles = pd.concat([profiles, prospect_profiles], ignore_index=True)
PROSPECT_EXCLUDED = set(EXCLUDED) | {"Prospects Challenge", "International-Jr"}
prospects, steps = prospect_table(drafts, all_profiles, set(projections["playerId"]), excluded=PROSPECT_EXCLUDED)
print("How the list was built:")
print(steps.to_string())
print("\nProspects by position:", prospects["group"].value_counts().reindex(["F", "D", "G"]).to_dict())
print("By draft year:", prospects["draft_year"].value_counts().sort_index().to_dict())

How the list was built:
drafted picks with a playerId                                  1777
  no profile (download missing or failed)                         0
  already in the NHL projection (played 2023-24 to 2025-26)     398
  didn't play a regular-season game in 2025-26                   26
  older than 24 on Feb 1, 2027                                  185
prospects                                                      1167

Prospects by position: {'F': 639, 'D': 389, 'G': 139}
By draft year: {2020: 63, 2021: 135, 2022: 146, 2023: 181, 2024: 204, 2025: 215, 2026: 223}


In [ ]:
# Leagues prospects played in during 2025-26 that look like short events (median under 15 games) and aren't on
# notebook 16's exclusion list: check none of these is really a tournament
p25 = all_profiles[(all_profiles["season"] == 2025) & (all_profiles["gameTypeId"] == 2)
                   & all_profiles["playerId"].isin(prospects["playerId"]) & ~all_profiles["league"].isin(PROSPECT_EXCLUDED)]
lg = p25.groupby("league").agg(players=("playerId", "nunique"), median_gp=("gp", "median"))
print("Prospects' 2025-26 leagues with a median under 15 games:")
lg[lg["median_gp"] < 15].sort_values("players", ascending=False)

Prospects' 2025-26 leagues with a median under 15 games:


,players,median_gp
league,,
U20 SM-sarja,28,14.0
U18 Region,15,3.0
Svenska Cupen U20,13,1.0
Mestis,12,7.0
U18 Nationell,11,3.0
Czechia2,10,4.0
U18 SM-sarja,7,4.0
Russia U18,5,3.0
DEL,5,13.0


## 4. Summarizing pre-NHL scoring: four rules

NHLe turns a season in another league into NHL points per game (notebook 16). A prospect has several pre-NHL seasons, so which one (or which mix) should stand for him? Four rules, tested the way notebook 16 tested its factors: predict each skater's **first NHL season with 20+ games** (points per game), with factors fit on data through 2017-18, scored on first NHL seasons from **2018-19 on** (games-weighted RMSE).

| Rule | What it uses |
|---|---|
| (i) | his best season with 20+ games (notebook 18's original rule) |
| (ii) | his best season with 10+ games, after pulling it toward his other pre-NHL seasons (or the league's average if he has none) with 20 games of weight, so a short hot streak doesn't count as much as a full season |
| (iii) | his most recent season with 20+ games |
| (iv) | the games-weighted NHLe of his last two pre-NHL seasons (10+ games each) |

All four are scored on the same players (those every rule can rate).

In [ ]:
f_train, m_train = training_nhle(careers, TRAIN_END)
lg_avg = league_average_nhle(careers, f_train, m_train, TRAIN_END)
rules = nhle_rules(careers, f_train, m_train, lg_avg)
RULES = {"i": "(i) best season, 20+ GP", "ii": "(ii) best 10+ GP season, shrunk", "iii": "(iii) most recent, 20+ GP",
         "iv": "(iv) last two seasons, weighted"}

nhl = careers[(careers["league"] == "NHL") & (careers["gp"] >= 20)].sort_values("season").groupby("playerId").head(1)
rk_pts = nhl.set_index("playerId").join(rules, how="inner")
rk_pts["ppg"] = rk_pts["points"] / rk_pts["gp"]
rows = []
for period, sel in [("test (2018-19 on)", rk_pts["season"] >= 2018), ("train (2006-07 to 2017-18)", rk_pts["season"].between(2006, 2017))]:
    d = rk_pts[sel]
    common = d[list(RULES)].notna().all(axis=1)
    for r, name in RULES.items():
        e = d.loc[common, r] - d.loc[common, "ppg"]
        rows.append({"period": period, "rule": name, "rookies": int(common.sum()), "coverage": d[r].notna().mean(),
                     "RMSE": np.sqrt(np.average(e ** 2, weights=d.loc[common, "gp"])),
                     "bias (pred - actual)": np.average(e, weights=d.loc[common, "gp"])})
rule_tbl = pd.DataFrame(rows).set_index(["period", "rule"])
test_err = rule_tbl.loc["test (2018-19 on)", "RMSE"]
WIN = [k for k, v in RULES.items() if v == test_err.idxmin()][0]
print(f"Winner: {RULES[WIN]} ({test_err.min():.4f}); next best {test_err.drop(RULES[WIN]).min():.4f} "
      f"({test_err.drop(RULES[WIN]).min() / test_err.min() - 1:.1%} worse); notebook 18's original rule (i) {test_err[RULES['i']]:.4f}")
rule_tbl.round(4)

Winner: (iv) last two seasons, weighted (0.1417); next best 0.1461 (3.1% worse); notebook 18's original rule (i) 0.1734


rookies  coverage    RMSE  bias (pred - actual)
period                     rule                                                                            
test (2018-19 on)          (i) best season, 20+ GP              694       1.0  0.1734                0.0852
                           (ii) best 10+ GP season, shrunk      694       1.0  0.1461                0.0326
                           (iii) most recent, 20+ GP            694       1.0  0.1461                0.0092
                           (iv) last two seasons, weighted      694       1.0  0.1417               -0.0041
train (2006-07 to 2017-18) (i) best season, 20+ GP             1130       1.0  0.1687                0.0662
                           (ii) best 10+ GP season, shrunk     1130       1.0  0.1513                0.0210
                           (iii) most recent, 20+ GP           1130       1.0  0.1465               -0.0047
                           (iv) last two seasons, weighted     1130       1.0  0.1433               -0.0198

In [ ]:
print("Matthew Schaefer (17 OHL games in his draft year) under each rule, NHL points per game:")
print(rules.loc[[8485366]].rename(columns=RULES).round(3).to_string())

Matthew Schaefer (17 OHL games in his draft year) under each rule, NHL points per game:
          (i) best season, 20+ GP  (ii) best 10+ GP season, shrunk  (iii) most recent, 20+ GP  (iv) last two seasons, weighted
playerId                                                                                                                      
8485366                     0.113                            0.282                      0.113                            0.198


The winner becomes an option of `development.best_nhle` (`rule="last2"`); the default stays the best-season rule, so notebook 17 reproduces exactly. (Notebook 18 has since switched to the new rule.)

**Does the better rule help NHL players too?** Notebook 18's chosen backtest (every skater gets the prior, 1,500 minutes, prior fit through 2017-18, test 2018-19 on), rerun with the new rule. Notebook 18 and `projections_2026-27.csv` are **not** changed here (notebook 18 was switched to the new rule afterwards).

In [ ]:
pairs = pd.concat([candidates(history, TARGETS + [2026]),
                   skaters[skaters["season"].isin(TARGETS)][["playerId", "season", "group"]]]).drop_duplicates(["playerId", "season"])
nhle_by_rule = {"i": best_nhle(careers, f_train, m_train), "iv": best_nhle(careers, f_train, m_train, rule="last2")}
feat = {r: skater_features(pairs, history, bios, profiles, n) for r, n in nhle_by_rule.items()}
info = feat["i"][["playerId", "season", "age", "experience"]]

def nb18_backtest(rule):
    tr = fit_rows(history, feat[rule], TRAIN)
    model = fit_prior(tr)
    cand = candidates(history, TEST).merge(feat[rule].drop(columns="group"), on=["playerId", "season"], how="left")
    pri = cand[["playerId", "season"]].assign(prior_gar_60=apply_prior(cand, model).to_numpy())
    b = backtest(skaters, TEST, SK_WEIGHTS, SK_REG, 300, bios, aging=AGING, priors=pri, prior_reg=1500)
    b = b.merge(info, on=["playerId", "season"], how="left")
    rm = lambda m: np.sqrt(np.average((b["actual"][m] - b["pred"][m]) ** 2, weights=b["toi"][m]))
    return {"all skaters": rm(b["age"] > 0), "23 and under": rm(b["age"] <= 23), "veterans (4+)": rm(b["experience"] >= 4)}

nb18_cmp = pd.DataFrame({f"rule {RULES[r]}": nb18_backtest(r) for r in ["i", WIN]}).T
print("Notebook 18's chosen backtest (GAR/60 RMSE, test 2018-19 on); rule (i) is notebook 18's original rule (0.2670 / 0.2834 / 0.2612):")
nb18_cmp.round(4)

Notebook 18's chosen backtest (GAR/60 RMSE, test 2018-19 on); rule (i) is notebook 18's original rule (0.2670 / 0.2834 / 0.2612):


,all skaters,23 and under,veterans (4+)
"rule (i) best season, 20+ GP",0.2670,0.2834,0.2612
"rule (iv) last two seasons, weighted",0.2665,0.2815,0.2610


## 5. The prior for players with no NHL time

A rookie has no NHL seasons, so his projection **is** his prior. Tested on notebook 18's true rookies: skaters whose first season in our NHL data is 2018-19 or later, with 300+ minutes (priors fit on target seasons through 2017-18; the training rookies 2009-10 to 2017-18 are shown too, in-sample). Five candidates:

- the **position average** (what notebook 15 would use),
- notebook 18's prior **(a)** (fit on every skater) with rule (i), and with the new rule,
- prior **(b)** (fit only on skaters with 3 or fewer NHL seasons) with the new rule,
- (b) for defensemen **without the draft terms** (notebook 18 found them wrong-signed for defensemen).

In [ ]:
first = first_seasons(history)
PRIORS = {"(a), rule (i)": ("a", "i", PRIOR_INPUTS), f"(a), rule ({WIN})": ("a", WIN, PRIOR_INPUTS),
          f"(b), rule ({WIN})": ("b", WIN, PRIOR_INPUTS),
          f"(b), D no draft, rule ({WIN})": ("b", WIN, ["nhle_ppg", "age", "experience"])}

def make_model(rows, name):
    """Fit one candidate prior. The 'D no draft' version drops the draft terms for defensemen only;
    its forwards use the full (b) model."""
    v, r, inputs = PRIORS[name]
    rows = rows[eligible(rows, v)]
    m = fit_prior(rows, inputs)
    if "D no draft" in name:
        m = {"F": fit_prior(rows[rows["group"] == "F"], PRIOR_INPUTS)["F"], "D": m["D"]}
    return m

models = {name: make_model(fit_rows(history, feat[r], TRAIN), name) for name, (v, r, inputs) in PRIORS.items()}

def rookies(seasons, rule):
    rk = skaters[(skaters["season"] == skaters["playerId"].map(first)) & skaters["season"].isin(seasons)
                 & skaters["season"].isin(TARGETS) & (skaters["toi"] >= 300)]
    rk = rk.assign(gar_60=rk["gar"] / rk["toi"] * 60).merge(feat[rule].drop(columns="group"), on=["playerId", "season"], how="left")
    rk = rk.merge(nhle_by_rule[WIN][["playerId", "nhle_league"]], on="playerId", how="left")
    rk["league group"] = rk["nhle_league"].map(league_group)
    rk["position average"] = [group_average(history, s - 1)[g] for s, g in zip(rk["season"], rk["group"])]
    for name, (v, r, inputs) in PRIORS.items():
        x = rk if r == rule else rk.drop(columns=["nhle_ppg"]).merge(feat[r][["playerId", "season", "nhle_ppg"]], on=["playerId", "season"], how="left")
        rk[name] = apply_prior(x, models[name]).to_numpy()
    return rk

rk = {"test": rookies(range(2018, 2026), "i"), "train": rookies(range(2009, 2018), "i")}
CANDS = ["position average"] + list(PRIORS)
rows = []
for period, d0 in rk.items():
    for c in CANDS:
        for g in ["F", "D"]:
            d = d0[d0["group"] == g]
            e = d[c] - d["gar_60"]
            rows.append({"period": period, "prior": c, "group": g, "rookies": len(d),
                         "RMSE (GAR/60)": np.sqrt(np.average(e ** 2, weights=d["toi"])),
                         "bias (pred - actual, WAR/60)": np.average(e, weights=d["toi"]) / GOALS_PER_WIN})
rookie_tbl = pd.DataFrame(rows).set_index(["period", "group", "prior"]).sort_index()
rookie_tbl.round(4)

rookies  RMSE (GAR/60)  bias (pred - actual, WAR/60)
period group prior                                                                           
test   D     (a), rule (i)                    86         0.2253                        0.0026
             (a), rule (iv)                   86         0.2222                        0.0029
             (b), D no draft, rule (iv)       86         0.2225                        0.0036
             (b), rule (iv)                   86         0.2240                        0.0033
             position average                 86         0.2091                       -0.0004
       F     (a), rule (i)                   121         0.3342                        0.0106
             (a), rule (iv)                  121         0.3353                        0.0110
             (b), D no draft, rule (iv)      121         0.3301                        0.0080
             (b), rule (iv)                  121         0.3301                        0.0080
             position average                121         0.3481                        0.0076
train  D     (a), rule (i)                   139         0.2127                        0.0008
             (a), rule (iv)                  139         0.2095                        0.0019
             (b), D no draft, rule (iv)      139         0.2068                        0.0015
             (b), rule (iv)                  139         0.2073                        0.0016
             position average                139         0.2175                       -0.0012
       F     (a), rule (i)                   196         0.3168                        0.0048
             (a), rule (iv)                  196         0.3179                        0.0061
             (b), D no draft, rule (iv)      196         0.3140                        0.0021
             (b), rule (iv)                  196         0.3140                        0.0021
             position average                196         0.3331                        0.0017

In [ ]:
t = rookie_tbl.loc["test"]
CHOSEN = {}
for g in ["F", "D"]:
    errs = t.loc[g, "RMSE (GAR/60)"]
    if g == "F":                    # the 'D no draft' version is the same model as (b) for forwards
        errs = errs.drop(f"(b), D no draft, rule ({WIN})")
    best = errs.drop("position average").idxmin()
    CHOSEN[g] = best if errs[best] < errs["position average"] else "position average"
    print(f"{'Forwards' if g == 'F' else 'Defensemen'}: best prior {best} ({errs[best]:.4f}) vs. position average "
          f"({errs['position average']:.4f}) -> use {CHOSEN[g]}")
if CHOSEN["D"] == "position average":
    print("No prior beats the position average for defensemen on the test seasons, so defensemen with no NHL time get "
          "the position average. Their (b, no draft) prior is kept as a ranking column only.")

Forwards: best prior (b), rule (iv) (0.3301) vs. position average (0.3481) -> use (b), rule (iv)
Defensemen: best prior (a), rule (iv) (0.2222) vs. position average (0.2091) -> use position average
No prior beats the position average for defensemen on the test seasons, so defensemen with no NHL time get the position average. Their (b, no draft) prior is kept as a ranking column only.


In [ ]:
# Bias by pre-NHL league group (the league of his most recent pre-NHL season), training vs. test,
# for the chosen prior of each position. A league adjustment is allowed only where both periods agree in sign.
rows = []
for g in ["F", "D"]:
    c = CHOSEN[g]
    for period, d0 in rk.items():
        d0 = d0[d0["group"] == g]
        for lgname, d in d0.groupby("league group"):
            rows.append({"group": g, "league group": lgname, "period": period, "rookies": len(d),
                         "bias": np.average(d[c] - d["gar_60"], weights=d["toi"]) / GOALS_PER_WIN})
lg_bias = pd.DataFrame(rows).pivot_table(index=["group", "league group"], columns="period", values=["rookies", "bias"])
lg_bias.columns = [f"{a} ({b})" for a, b in lg_bias.columns]
lg_bias["same sign"] = np.sign(lg_bias["bias (train)"]) == np.sign(lg_bias["bias (test)"])
lg_bias["enough rookies"] = (lg_bias["rookies (train)"] >= 5) & (lg_bias["rookies (test)"] >= 5)
lg_bias["adjust"] = lg_bias["same sign"] & lg_bias["enough rookies"]
print("Bias (predicted minus actual, WAR/60; positive = over-predicted) by pre-NHL league group:")
lg_bias.round(4)

Bias (predicted minus actual, WAR/60; positive = over-predicted) by pre-NHL league group:


bias (test)  bias (train)  rookies (test)  rookies (train)  same sign  enough rookies  adjust
group league group                                                                                               
D     AHL                0.0037        0.0046            38.0             50.0       True            True    True
      CHL               -0.0004       -0.0073            15.0             29.0       True            True    True
      Europe            -0.0039        0.0063            17.0             19.0      False            True   False
      KHL               -0.0130        0.0177             7.0             10.0      False            True   False
      NCAA               0.0007       -0.0104             9.0             30.0      False            True   False
      other                 NaN       -0.0133             NaN              1.0      False           False   False
F     AHL               -0.0072       -0.0083            42.0             57.0       True            True    True
      CHL                0.0329        0.0157            27.0             63.0       True            True    True
      Europe             0.0082        0.0001            20.0             34.0       True            True    True
      KHL               -0.0113       -0.0099            17.0             10.0       True            True    True
      NCAA               0.0116       -0.0095            14.0             32.0      False            True   False
      other              0.0304           NaN             1.0              NaN      False           False   False

In [ ]:
# Does subtracting the training bias of the qualifying league groups help on the test rookies?
# They're used for a position only if they lower its test error.
adj = lg_bias.loc[lg_bias["adjust"], "bias (train)"]
ADJ_USE = {}
for g in ["F", "D"]:
    d = rk["test"][rk["test"]["group"] == g]
    base = d[CHOSEN[g]] - d["gar_60"]
    shift = pd.Series([adj.get((g, x), 0.0) for x in d["league group"]], index=d.index) * GOALS_PER_WIN
    after = base - shift
    ADJ_USE[g] = np.average(after ** 2, weights=d["toi"]) < np.average(base ** 2, weights=d["toi"])
    print(f"{'used' if ADJ_USE[g] else 'NOT used'} for {g}: adjusted groups {[x for (gg, x) in adj.index if gg == g] or 'none'}; test RMSE "
          f"{np.sqrt(np.average(base ** 2, weights=d['toi'])):.4f} -> {np.sqrt(np.average(after ** 2, weights=d['toi'])):.4f}")

used for F: adjusted groups ['AHL', 'CHL', 'Europe', 'KHL']; test RMSE 0.3301 -> 0.3171
NOT used for D: adjusted groups ['AHL', 'CHL']; test RMSE 0.2091 -> 0.2095


In [ ]:
# Rookie range check: spread of test-rookie misses under the chosen prior, in notebook 17's scaled units,
# next to the rookie rows saved by notebook 18 (measured on notebook 18's prior)
rows = []
for g in ["F", "D"]:
    d = rk["test"][rk["test"]["group"] == g]
    miss = (d["gar_60"] - d[CHOSEN[g]]) / GOALS_PER_WIN
    z = miss / np.sqrt(luck.loc[g, "a"] + luck.loc[g, "b"] / d["toi"])
    zc = z - wmean(z, d["toi"])
    saved = rookie_range.set_index("group").loc[g, "z_sd"]
    rows.append({"group": g, "z SD, chosen prior": wsd(zc.to_numpy(), d["toi"].to_numpy()), "z SD, saved rookie rows": saved})
zc_tbl = pd.DataFrame(rows).set_index("group")
zc_tbl["flag"] = (zc_tbl["z SD, chosen prior"] - zc_tbl["z SD, saved rookie rows"]).abs() > 0.1
ROOKIE_MINUTES = float(np.median(pd.concat(rk.values())["toi"]))
print(f"Typical true-rookie season: {ROOKIE_MINUTES:,.0f} minutes (median); the prospect ranges below use it.")
zc_tbl.round(3)

Typical true-rookie season: 651 minutes (median); the prospect ranges below use it.


,"z SD, chosen prior","z SD, saved rookie rows",flag
group,,,
F,0.941,0.948,False
D,1.019,1.100,False


## 6. Goalies

There's no NHLe for goalies (they don't score), so goalie prospects keep the **league-average goalie** prior from 2025-26 and are flagged. That is almost certainly too generous: a goalie's first NHL season is usually below average. Phase 2 has to handle rookie goalies (for example with a rookie-goalie discount fit on first seasons).

## 7. The 2026-27 prospect projections

The final priors are refit on **all** target seasons with notebook 16's whole-window factors and the new pre-NHL rule. Each prospect's age (February 1, 2027), NHL experience (almost always 0) and draft position feed the chosen prior for his position. The projection means **how good he'd be per 60 minutes if he plays NHL games in 2026-27**, not that he will.

Ranges: the 10th and 90th percentile WAR/60 at a typical rookie's minutes, from the rookie rows in `dev_ranges.csv` (notebook 18). Goalies get no range.

**Two fixes for prospects only** (NHL players and notebooks 16-18 are unchanged):
- **Renamed junior leagues.** Sweden's top junior league is listed as "U20 Nationell" from 2025-26 on (it was "J20 Nationell"), and Finland's "U20 SM-liiga" (2015-16 to 2019-20) as "U20 SM-sarja" from 2020-21 on. Same leagues, new names, so a prospect's seasons there get the old name's factor. (A handful of pre-2015 "U20 SM-sarja" seasons belong to an older, different series and keep their name.)
- **No NHLe at all.** A prospect whose recent seasons are all in leagues without a factor used to get his position's *average* NHLe, which flatters players from weak leagues. Now he gets the **25th percentile** of prospects' NHLe at his position: a cautious guess, since a player whose scoring we can't rate is more likely to come from a lower league. Flagged in `nhle_missing`.

In [ ]:
# Profiles, bios and careers for everyone: NHL players plus the downloaded prospects
bios_all = pd.concat([bios, build_bio_table(RAW_PROSPECT, 2026)]).drop_duplicates("playerId") if any(RAW_PROSPECT.glob("*.json")) else bios
careers_all = career_totals(all_profiles)
nhle_full_i = best_nhle(careers_all, nhle_factors, nhle_mult)

# League renames, prospects only
def prospect_league(league, season):
    if league == "U20 Nationell":
        return "J20 Nationell"
    if league == "U20 SM-sarja" and season >= 2020:
        return "U20 SM-liiga"
    return league

sk_ids = set(prospects.loc[prospects["group"] != "G", "playerId"])
no_nhle = lambda n: len(sk_ids - set(n["playerId"]))
before = no_nhle(best_nhle(careers_all, nhle_factors, nhle_mult, rule="last2"))
is_p = careers_all["playerId"].isin(prospects["playerId"])
careers_all.loc[is_p, "league"] = [prospect_league(l, s) for l, s in zip(careers_all.loc[is_p, "league"], careers_all.loc[is_p, "season"])]
first_cols = ["name", "position", "group", "birth_date", "age"]
careers_all = (careers_all.groupby(["playerId", "season", "league"], as_index=False)
               .agg(**{c: (c, "first") for c in first_cols}, **{c: (c, "sum") for c in ["gp", "goals", "assists", "points"]}))
nhle_full = best_nhle(careers_all, nhle_factors, nhle_mult, rule="last2")
print(f"Skater prospects with no NHLe: {before} before the renames, {no_nhle(nhle_full)} after")

# Final priors: refit on all target seasons
feat_full = skater_features(pairs, history, bios, profiles, best_nhle(careers, nhle_factors, nhle_mult, rule="last2"))
rows_full = fit_rows(history, feat_full, TARGETS)
final = {}
for name in set(CHOSEN.values()) | {f"(b), D no draft, rule ({WIN})"}:
    if name == "position average":
        continue
    final[name] = make_model(rows_full, name)
for name, m in final.items():
    print(f"Final prior {name}:")
    print(model_table(m).round(4).to_string(), "\n")

sk_p = prospects[prospects["group"] != "G"][["playerId", "group"]].assign(season=2026)
pf = skater_features(sk_p, history, bios_all, all_profiles, nhle_full).merge(
    nhle_full[["playerId", "nhle_league", "nhle_season"]], on="playerId", how="left")
pf["nhle_missing"] = pf["nhle_ppg"].isna()
fill25 = pf[~pf["nhle_missing"]].groupby("group")["nhle_ppg"].quantile(0.25)
pf["nhle_filled"] = pf["nhle_ppg"].fillna(pf["group"].map(fill25))
print(f"Skater prospects with no NHLe: {pf['nhle_missing'].sum()} "
      f"({', '.join(f'{g} {n}' for g, n in pf.groupby('group')['nhle_missing'].sum().items())}); "
      f"filled with the 25th percentile: " + ", ".join(f"{g} {v:.3f}" for g, v in fill25.items()) + " NHL points per game")
avg25 = group_average(history, 2025)
proj = prospects.merge(pf[["playerId", "experience", "nhle_ppg", "nhle_missing", "nhle_league", "nhle_season"]], on="playerId", how="left")
proj["league_last"] = [prospect_league(l, 2025) for l in proj["league_last"]]
proj["prior_type"] = proj["group"].map({"F": CHOSEN["F"], "D": CHOSEN["D"], "G": "league-average goalie (flagged)"})
proj["proj_war_60"] = np.nan
for g in ["F", "D"]:
    sel = proj["group"] == g
    x_old = pf.set_index("playerId").loc[proj.loc[sel, "playerId"]].reset_index()  # missing NHLe -> position average (old)
    x = x_old.assign(nhle_ppg=x_old["nhle_filled"])                                 # missing NHLe -> 25th percentile
    if CHOSEN[g] == "position average":
        proj.loc[sel, "proj_war_60"] = avg25[g] / GOALS_PER_WIN
    else:
        proj.loc[sel, "proj_war_60"] = apply_prior(x, final[CHOSEN[g]]).to_numpy() / GOALS_PER_WIN
        proj.loc[sel, "proj_old_fill"] = apply_prior(x_old, final[CHOSEN[g]]).to_numpy() / GOALS_PER_WIN
    if g == "D":
        proj.loc[sel, "d_rank_prior"] = apply_prior(x, final[f"(b), D no draft, rule ({WIN})"]).to_numpy() / GOALS_PER_WIN
    # league adjustment for qualifying groups (training bias), only where it helped on the test rookies
    if ADJ_USE[g]:
        shift = pd.Series([adj.get((g, league_group(l)), 0.0) for l in proj.loc[sel, "nhle_league"]], index=proj.index[sel])
        proj.loc[sel, "proj_war_60"] -= shift
        if "proj_old_fill" in proj:
            proj.loc[sel, "proj_old_fill"] -= shift
proj.loc[proj["group"] == "G", "proj_war_60"] = avg25["G"] / GOALS_PER_WIN
for g in ["F", "D"]:
    sel = proj["group"] == g
    rng = prospect_range(proj.loc[sel, "proj_war_60"], g, rookie_range, luck, ROOKIE_MINUTES)
    for k, v in rng.items():
        proj.loc[sel, k] = v
print(f"Projected {len(proj):,} prospects. Forwards: prior {CHOSEN['F']}; defensemen: {CHOSEN['D']}; goalies: league average.")
print(f"Average projection (WAR/60): " + ", ".join(f"{g} {proj.loc[proj['group'] == g, 'proj_war_60'].mean():.4f}" for g in ["F", "D", "G"]))
moved = proj[proj["nhle_missing"].eq(True) & (proj["group"] == "F")]
print(f"\nForwards with no NHLe, projection with the old fill (position average) vs. the new (25th percentile):")
print(moved[["name", "draft_year", "pick", "team", "league_last", "proj_old_fill", "proj_war_60"]]
      .sort_values("pick").round(3).to_string(index=False))

Skater prospects with no NHLe: 26 before the renames, 6 after
Final prior (b), rule (iv):
                 D               F        
              coef      se    coef      se
constant   -0.0347  0.0690  0.1660  0.0913
nhle_ppg    0.5585  0.0543  0.9879  0.0614
log_pick   -0.0034  0.0057 -0.0010  0.0068
undrafted  -0.0060  0.0306 -0.0153  0.0397
age        -0.0002  0.0033 -0.0129  0.0043
experience  0.0044  0.0058  0.0308  0.0074 

Final prior (b), D no draft, rule (iv):
                 F               D        
              coef      se    coef      se
constant    0.1660  0.0913 -0.0356  0.0620
nhle_ppg    0.9879  0.0614  0.5650  0.0537
log_pick   -0.0010  0.0068     NaN     NaN
undrafted  -0.0153  0.0397     NaN     NaN
age        -0.0129  0.0043 -0.0007  0.0025
experience  0.0308  0.0074  0.0048  0.0054 

Skater prospects with no NHLe: 6 (D 3, F 3); filled with the 25th percentile: D 0.087, F 0.156 NHL points per game
Projected 1,167 prospects. Forwards: prior (b), rule (iv); defe

In [ ]:
COLS = ["playerId", "name", "position", "group", "draft_year", "pick", "round", "team", "age", "league_last",
        "gp_last", "points_last", "nhle_ppg", "nhle_missing", "nhle_league", "prior_type", "proj_war_60", "war_60_p10", "war_60_p50",
        "war_60_p90", "d_rank_prior"]
out = proj[COLS].rename(columns={"team": "draft_team", "league_last": "league_2025_26", "gp_last": "gp_2025_26",
                                 "points_last": "points_2025_26", "nhle_ppg": "nhle_last2"})
out.to_csv(OUT / "prospects_2026-27.csv", index=False)
has_factor = set(nhle_factors.loc[nhle_factors["chosen"], "league"])
no_factor = out[(out["group"] != "G") & ~out["league_2025_26"].isin(has_factor)]["league_2025_26"].value_counts()
print("Skater prospects whose 2025-26 league has no NHLe factor (their NHLe comes from other seasons, or the "
      "position average if none):", no_factor.to_dict() or "none")
print(f"Saved prospects_2026-27.csv ({len(out):,} prospects). d_rank_prior is for ranking defensemen only "
      "(their projection is the position average).")

Skater prospects whose 2025-26 league has no NHLe factor (their NHLe comes from other seasons, or the position average if none): {'USports': 10, 'HockeyEttan': 8, 'Slovakia': 5, 'Czechia2': 1, 'Germany3': 1, 'ACH': 1, 'Belarus': 1, 'Division 2': 1, 'U18 AAA': 1, 'GOJHL': 1, 'Czechia U20': 1}
Saved prospects_2026-27.csv (1,167 prospects). d_rank_prior is for ranking defensemen only (their projection is the position average).


## 8. Hockey checks

Projections should roughly follow draft position and recent scoring. The top prospects by projection, the 2026 first round's top 10 in draft order, the Red Wings' prospects, and Schaefer under the new rule.

In [ ]:
SHOW = ["name", "position", "draft_year", "pick", "draft_team", "age", "league_2025_26", "gp_2025_26",
        "points_2025_26", "nhle_last2", "proj_war_60", "war_60_p10", "war_60_p90", "d_rank_prior"]
print("Top 25 prospects by projection (forwards; defensemen all share the position average):")
print(out[out["group"] == "F"].nlargest(25, "proj_war_60")[SHOW].round(3).to_string(index=False))
print("\nTop 15 defensemen by the ranking prior:")
print(out[out["group"] == "D"].nlargest(15, "d_rank_prior")[SHOW].round(3).to_string(index=False))

Top 25 prospects by projection (forwards; defensemen all share the position average):
               name position  draft_year  pick draft_team  age league_2025_26  gp_2025_26  points_2025_26  nhle_last2  proj_war_60  war_60_p10  war_60_p90  d_rank_prior
      Gavin McKenna       LW        2026     1        TOR 19.0           NCAA        35.0            51.0       0.686        0.100       0.032       0.179           NaN
    Roman Kantserov       RW        2023    44        CHI 22.0            KHL        63.0            64.0       0.632        0.094       0.026       0.173           NaN
    Andrew Cristall       LW        2023    40        WSH 21.0            AHL        72.0            60.0       0.530        0.078       0.010       0.157           NaN
   Ethan Wyttenbach       LW        2025   144        CGY 19.0           NCAA        40.0            59.0       0.510        0.070       0.002       0.149           NaN
         Max Plante       LW        2024    47        DET 20.0       

In [ ]:
top10 = drafts[(drafts["draft_year"] == 2026) & (drafts["pick"] <= 10)][["pick", "playerId", "name", "position", "team", "amateur_league"]]
top10 = top10.merge(out[["playerId", "league_2025_26", "gp_2025_26", "points_2025_26", "nhle_last2", "proj_war_60", "d_rank_prior"]]
                    .assign(prospect=True), on="playerId", how="left")
top10["prospect"] = top10["prospect"].eq(True)
print("2026 first round, top 10, in draft order (prospect = False means not on the prospect list; see section 3):")
top10.drop(columns="playerId").round(3)

2026 first round, top 10, in draft order (prospect = False means not on the prospect list; see section 3):


,pick,name,position,team,amateur_league,league_2025_26,gp_2025_26,points_2025_26,nhle_last2,proj_war_60,d_rank_prior,prospect
0,1,Gavin McKenna,LW,TOR,BIG10,NCAA,35.0,51.0,0.686,0.100,NaN,True
1,2,Ivar Stenberg,LW,SJS,SWEDEN,SHL,43.0,33.0,0.330,0.041,NaN,True
2,3,Caleb Malhotra,C,VAN,OHL,OHL,67.0,84.0,0.305,0.023,NaN,True
3,4,Daxon Rudolph,D,BUF,WHL,WHL,68.0,78.0,0.290,0.023,0.019,True
4,5,Alberts Smits,D,NYR,GERMANY,Liiga,38.0,13.0,0.136,0.023,0.005,True
5,6,Carson Carels,D,CGY,WHL,WHL,58.0,73.0,0.294,0.023,0.020,True
6,7,Chase Reid,D,SEA,OHL,OHL,45.0,48.0,0.344,0.023,0.024,True
7,8,Viggo Bjorck,C,WPG,SWEDEN,SHL,42.0,15.0,0.287,0.036,NaN,True
8,9,Keaton Verhoeff,D,SJS,NCHC,NCAA,36.0,20.0,0.255,0.023,0.016,True
9,10,Wyatt Cullen,LW,NSH,NTDP - USHL,USHL,15.0,16.0,0.249,0.030,NaN,True


In [ ]:
print("Red Wings prospects (drafted by DET):")
det = out[out["draft_team"] == "DET"].sort_values(["group", "proj_war_60", "d_rank_prior"], ascending=[False, False, False])
print(det[SHOW].round(3).to_string(index=False))

Red Wings prospects (drafted by DET):
                    name position  draft_year  pick draft_team  age    league_2025_26  gp_2025_26  points_2025_26  nhle_last2  proj_war_60  war_60_p10  war_60_p90  d_rank_prior
              Jan Bednar        G        2020   107        DET 24.0             Liiga        44.0             0.0         NaN        0.046         NaN         NaN           NaN
          Trey Augustine        G        2023    41        DET 21.0              NCAA        34.0             0.0         NaN        0.046         NaN         NaN           NaN
            Rudy Guimond        G        2023   169        DET 21.0             QMJHL        50.0             0.0         NaN        0.046         NaN         NaN           NaN
           Landon Miller        G        2024   126        DET 21.0               OHL        33.0             0.0         NaN        0.046         NaN         NaN           NaN
           Michal Pradel        G        2025    75        DET 19.0          

In [ ]:
sch = 8485366
print("Matthew Schaefer (in the NHL projection, not a prospect):")
print(f"  NHLe, notebook 18's original rule (best 20+ GP season): {nhle_full_i.set_index('playerId').loc[sch, 'nhle_ppg']:.3f}")
print(f"  NHLe, new rule (last two seasons):                      {nhle_full.set_index('playerId').loc[sch, 'nhle_ppg']:.3f}")
p18 = projections.set_index("playerId").loc[sch]
s_feat = feat_full[(feat_full["playerId"] == sch) & (feat_full["season"] == 2026)]
print(f"  Notebook 18 prior (from projections_2026-27.csv): {p18['prior_war_60']:.3f} WAR/60 (projection {p18['new_proj_war_60']:.3f})")
print(f"  (b, D no draft) prior with the new rule: {apply_prior(s_feat, final[f'(b), D no draft, rule ({WIN})']).iloc[0] / GOALS_PER_WIN:.3f} WAR/60")

Matthew Schaefer (in the NHL projection, not a prospect):
  NHLe, notebook 18's original rule (best 20+ GP season): 0.106
  NHLe, new rule (last two seasons):                      0.186
  Notebook 18 prior (from projections_2026-27.csv): 0.024 WAR/60 (projection 0.064)
  (b, D no draft) prior with the new rule: 0.010 WAR/60


## Takeaways
- **Summarize pre-NHL scoring with the last two seasons, not the best one.** On notebook 16's points backtest (first NHL seasons from 2018-19 on, factors fit through 2017-18), the games-weighted NHLe of a player's last two pre-NHL seasons has an error of 0.142 points per game, vs. 0.146 for the shrunk best season and for the most recent season, and 0.173 for notebook 18's original rule, the best season with 20+ games. The training seasons rank them the same way. Picking the best season picks lucky years: it over-predicts by 0.085 points per game, while the last two seasons are almost unbiased (-0.004). It's now `best_nhle(rule="last2")`; the default is unchanged, so notebook 17 reproduces exactly.
- **Schaefer's case is fixed:** his NHLe goes from 0.106 to 0.186 NHL points per game, because his 17-game draft year now counts. With notebook 18 on the new rule, his prior goes from 0.016 to 0.024 WAR/60.
- **The new rule helps NHL players too, a little.** Notebook 18's chosen backtest with it: 0.2665 vs. 0.2670 GAR/60 for all skaters, 0.2815 vs. 0.2834 for skaters 23 and under, 0.2610 vs. 0.2612 for veterans. **Done since:** notebook 18 (and `projections_2026-27.csv`) now use the new rule, so NHL players and prospects use the same one.
- **Forwards with no NHL time:** the prior fit on players with 3 or fewer NHL seasons, with the new rule, has an error of 0.330 GAR/60 on the 121 test rookies, vs. 0.348 for the position average (5% better).
  - **League adjustments:** AHL, CHL, Europe and KHL forwards show the same sign of bias in training and test.
    - CHL rookies are over-predicted (+0.016 WAR/60 training, +0.033 test), the same pattern as notebook 16.
    - AHL and KHL rookies are under-predicted.
    - Europe's training bias is near zero.
  - Subtracting the training biases lowers the test error to 0.317, so forwards get these adjustments.
- **Defensemen with no NHL time get the position average.** No prior beats it on the test rookies (best 0.222 vs. 0.209 GAR/60, 86 rookies), even though the priors win in training. Two league adjustments qualify (AHL, CHL) but they make the test error slightly worse, so they're left out. The (b, no draft terms, new rule) prior is saved as `d_rank_prior`, for ranking defensemen only.
- **The rookie range from notebook 18 still fits.** Test-rookie spread in notebook 17's scaled units is 0.94 for forwards (notebook 18's original rows: 0.95) and 1.02 for defensemen (1.12; 1.10 after notebook 18's switch), all within 0.1. Ranges use a typical rookie season of 651 minutes. The prospect ranges in this run use notebook 18's original rookie rows; a rerun would use the new ones (a small change).
- **Goalie prospects** get the league-average goalie (flagged). That's almost certainly too high for a rookie goalie, and Phase 2 has to handle it.
- **The prospect list:** 1,777 draft picks from 2019 to 2026 (one player was picked twice) become 1,167 prospects (639 forwards, 389 defensemen, 139 goalies) after removing 398 already in the NHL projection, 26 who didn't play in 2025-26 and 185 older than 24. Mapping Sweden's and Finland's renamed top junior leagues (U20 Nationell, U20 SM-sarja) cut skater prospects with no NHLe from 26 to 6; those 6 get the 25th percentile NHLe for their position (F 0.156, D 0.087) and are flagged `nhle_missing`.
- **Top prospects:** Gavin McKenna (2026 #1, 51 points in 35 NCAA games) leads at 0.100 WAR/60, then Roman Kantserov (KHL, 64 points in 63 games) 0.094, Andrew Cristall (AHL) 0.078, Ethan Wyttenbach (pick 144, 59 points in 40 NCAA games) 0.070 and Max Plante (NCAA) 0.067.
- **The 2026 top 10:** McKenna is far ahead. Stenberg (#2) projects 0.041, Bjorck (#8) 0.036, Cullen (#10) 0.030 and Malhotra (#3) 0.023. The five defensemen share the position average (0.023); by the ranking prior, Reid (0.024), Carels (0.020) and Rudolph (0.019) lead them.
- **CHL forwards rank below their scoring** because of the CHL adjustment (Jake O'Brien, 93 points, 0.059; Caleb Desnoyers 0.051; Malhotra 0.023). CHL rookies were over-predicted in both training and test, so the adjustment is backed by the data, but it's the biggest judgment call in the list.
- **For forwards with no NHL time, draft position adds nothing once NHLe is known** (log pick coefficient about 0). That's why pick 144 Wyttenbach projects above #2 Stenberg. Phase 2's call-up order should still weigh draft position.
- **Red Wings:** the top skater prospects are Max Plante (0.067) and Dmitri Buchelnikov (0.064), then Amadeus Lombardi (0.049), Carter Bear (0.043) and JP Hurlbert (2026 #23, 0.041). Their six goalie prospects, including Trey Augustine, all sit at the league-average goalie (0.046), flagged.
- **Next:** minutes projections (who plays, how much, and which prospects get called up) and the season simulator.